In [0]:
import requests
import datetime
import time
import logging
from pyspark.sql import Window
from delta.tables import DeltaTable
from pyspark.sql import functions as F

url = 'https://api.weather.gov/alerts/active'
headers = {'user-agent':"SwiftLogix_Data_Project"}

# response = requests.get(url, headers = headers)

# Configure logging for Databricks Driver logs
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("API_Pipeline")

# Fault Tolerance test
def call_api(url, headers):
  max_retries = 3
  backoff = 2

  for attempt in range(max_retries):
    try:
      response = requests.get(url, headers = headers, timeout = 30)
      response.raise_for_status() #if we dont use the raise_for_status() it will continue to run the rest of the code and not throw an error
                                  #even if the API failed. When an error occurs, it will trigger and except line below
      return response
    
    except requests.Timeout as t:
      if attempt < max_retries:
        wait = backoff * (attempt + 1)
        logger.warning(f'Attempt {attempt + 1} failed ({t}). Retrying in {wait} seconds...')
        time.sleep(wait)
      else:
        logger.error("Final retry attempt failed.")

    except requests.RequestException as e:
      logger.error(f"Request Exception: {e}")
      dbutils.notebook.exit(f'FAILED: API Request Exception {e}') #Exit the notebook with a message

  dbutils.notebook.exit(f'FAILED: API TIMED OUT AFTER {max_retries} ATTEMPTS')

response = call_api(url, headers)

#--TABLE PATH
catalog = 'bronze'
schema = 'swiftlogix'
table_name = 'bronze_weather'
full_table_name = f'{catalog}.{schema}.{table_name}'


#--As instructed we will just upload the newest data for this project. It doesnt seem like previous data will be needed for the analysis .
raw_json = spark.createDataFrame([response.text], ['raw_data'])
# raw_json = raw_json.withColumn('timestamp', F.lit(datetime.datetime.now()))
raw_json = raw_json.withColumn('timestamp', F.current_timestamp())

raw_json.write.mode('overwrite') \                    # medalion we use append not overwrite for the mode 
                .option('OverwriteSchema', 'true') \
                .saveAsTable(full_table_name)

In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType, MapType

temp_bronze_weather = spark.read.table(full_table_name)

#Should I incorporate the timestamp from the bronze or use a new one?
json_weather = json.loads(temp_bronze_weather.select('raw_data').first()['raw_data'])
weather_features = json_weather.get('features') # json_weather['features']

schema = StructType([
    StructField("id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("properties", MapType(StringType(), StringType()), True),
    StructField("geometry", MapType(StringType(), StringType()), True)
])

weather = spark.createDataFrame(weather_features, schema=schema)
weather = weather.withColumnRenamed('id', 'id_origin')
weather = weather.withColumn('record_id', F.row_number().over(Window.orderBy(F.lit(1))))
display(weather)
initial_row_count = weather.count()

#CHECK TO SEE IF THE COLUMN TYPE CAN BE DISCARTE OR NOT BASE ON HOW MANY DIFFERENT TYPE OF VALUES THERE ARE
# display(weather.select('type').distinct())
#CREATE A TEMP VIEW TO USE IN SQL
# weather.createOrReplaceTempView("weather")


In [0]:
from pyspark.sql.functions import explode, first

# Explode the properties map from the dataframe
weather_exploded = weather.select(
    'record_id',
    "id_origin",
    explode("properties").alias("property_key", "property_value")
)

# Pivot to make property_key values become columns
# For now I will assume id and @id are the same for their corresponding row,, thats why I will drop it
weather_pivoted = weather_exploded.groupBy('record_id',"id_origin").pivot("property_key").agg(first("property_value")).drop("@id")
display(weather_pivoted)


In [0]:
# Explode the geometry column from the dataframe
weather_exploded_polygon = weather.select(
    "id_origin",
    explode("geometry").alias("coordinate_index", "coordinate_value"),
)

weather_pivoted_poly = weather_exploded_polygon.groupBy("id_origin").pivot("coordinate_index").agg(first("coordinate_value"))

display(weather_pivoted_poly)

In [0]:
#Join the two tables
weather_silver = weather_pivoted.join(weather_pivoted_poly, on='id_origin', how='left')
display(weather_silver)
final_row_count = weather_silver.count()

#use logger to do the print statements
if initial_row_count == final_row_count:
  print("All rows were preserved")
else:
  print("Some rows were dropped")


In [0]:
#--In Databricks, Volumes are accessed via /Volumes
csv_path = '/Volumes/bronze/swiftlogix/ugc_with_location'

#--Need to remove any spaces from the columns names and repalce them with '_'
ugc_csv = spark.read.csv(csv_path, header=True, sep="|").toDF(*[col.replace(' ', '_').lower() for col in spark.read.csv(csv_path, header=True, sep="|").columns])
# display(ugc_csv)


#--Need to remove any spaces from the columns names and repalce them with '_')

test = weather_silver.withColumn('parsed_map', F.from_json('geocode', 'MAP<STRING,ARRAY<STRING>>'))
# display(test)

test2 = test.select('*', F.explode('parsed_map').alias('code_type','geocodes'))
# display(test2)

test3 = test2.groupBy('record_id').pivot('code_type').agg(F.first('geocodes'))
# display(test3)

test_same = test3.select('record_id', F.explode('SAME').alias('exploded_same')).drop('SAME')
test_same = test_same.withColumn('fips', F.substring('exploded_same', 2, 6))
test_ugc = test3.select('record_id', F.explode('UGC').alias('exploded_ugc')).drop('UGC')
test_ugc = test_ugc.withColumn('state', F.substring('exploded_ugc', 1, 2))
test_ugc = test_ugc.withColumn('zone', F.substring('exploded_ugc', 4, 5))

test_same_final = test_same.join(ugc_csv, (test_same['fips']==ugc_csv['FIPS']), how='inner').drop(ugc_csv['FIPS'])
test_same_final = test_same_final.withColumn('lat', F.col('lat').cast('double'))
test_same_final = test_same_final.withColumn('lon', F.col('lon').cast('double'))

# display(test_same)
# display(test_ugc)
display(test_same_final)


In [0]:
#Filter only for status = Actual and severity in ('Extreme', 'Severe', 'Moderate')
#Droping the columsn first will accelerate the process when filtering
weather_silver_fields = weather_silver.select('record_id', 'event', 'severity','status','headline','areaDesc','ends','expires')
#If there is no end time use the experires column as a reference
weather_silver_fields = weather_silver_fields.withColumn('ends', F.when(F.col('ends').isNull() | (F.col('ends') == "null"), 
                                                                        F.col('expires')).otherwise(F.col('ends'))).drop('expires')
weather_silver_fields = weather_silver_fields[(weather_silver_fields['status']=='Actual') & (weather_silver_fields['severity'].isin(['Extreme', 'Severe', 'Moderate']))]

display(weather_silver_fields)

silver_catalog = 'silver'
silver_schema = 'swiftlogix'
silver_table = 'silver_weather'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")

# merge statement / if id doesnt exist it will append 
# for fact tables use merge statement to for historical data (scd2 not necessary for this project)
weather_silver_fields.write \
    .format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable(f'{silver_catalog}.{silver_schema}.{silver_table}')


#--Read the csv file using the define table path
#--In Databricks, Volumes are accessed via /Volumes
csv_path = '/Volumes/bronze/swiftlogix/major_us_cities_location'

#--Need to remove any spaces from the columns names and repalce them with '_'
Csv_df = spark.read.csv(csv_path, header=True).toDF(*[col.replace(' ', '_') for col in spark.read.csv(csv_path, header=True).columns])
# display(Csv_df)

In [0]:
silver_weather_df = spark.read.table(f'{silver_catalog}.{silver_schema}.{silver_table}')

hazard_type_df = silver_weather_df.select('event').distinct()
hazard_type_df = hazard_type_df.withColumnRenamed('event', 'hazard_type')
hazard_type_df = hazard_type_df.withColumn('hazard_type_id', F.row_number().over(Window.orderBy('hazard_type'))).select('hazard_type_id', 'hazard_type')
display(hazard_type_df)

silver_weather_df = silver_weather_df.join(hazard_type_df, (silver_weather_df['event'] == hazard_type_df['hazard_type']), how='left').drop('hazard_type', 'event')
# display(silver_weather_df)

# ### USING THE AREDESC TO GET THE LOCATIONS
# sw_df_exploded = silver_weather_df.withColumn('areaDesc', F.explode(F.split(F.col('areaDesc'), ';')))
# # # silver_weather_df_exploded = silver_weather_df.withColumn('areaDesc', F.split(F.col('areaDesc'), ';')) #Test to see how it would look as an array
# # silver_weather_df_exploded = silver_weather_df_exploded.withColumn('state', F.col('headline').substr(-2, 2))
# sw_df_exploded = sw_df_exploded.withColumn('AD_cleaned', F.ltrim(F.col('areaDesc'))).drop('areaDesc')
# sw_df_exploded = sw_df_exploded.withColumn('array_cleaned',F.split(F.col('AD_cleaned'), ',')) \
#                                             .withColumn('array_size', F.size(F.col('array_cleaned'))).drop('AD_cleaned')

# sw_df_exploded2 = sw_df_exploded[sw_df_exploded['array_size']==2].withColumn('city', F.col('array_cleaned')[0]) \
#                                                                 .withColumn('state', F.col('array_cleaned')[1]).drop('array_cleaned','array_size') \
#                                                                 .withColumn('state', F.ltrim(F.col('state')))

# sw_df_exploded3 = sw_df_exploded2.filter(F.length(F.col('state'))==2).withColumn('operational_status', F.when(F.col('severity')=='Extreme', 'Suspended') \
#                                                                                                         .when(F.col('severity')=='Severe','Delayed') \
#                                                                                                         .otherwise('Normal'))

### USING THE SAME/UGC CODE TO GE THE LOCATIONS
sw_df_exploded3 = silver_weather_df.withColumn('operational_status', F.when(F.col('severity')=='Extreme', 'Suspended') \
                                                                                                        .when(F.col('severity')=='Severe','Delayed') \
                                                                                                        .otherwise('Normal'))

display(sw_df_exploded3)



In [0]:
#Join silver weather table and the warehouse table

# ### USING THE AREDESC TO GET THE LOCATIONS
Csv_df2 = Csv_df.withColumnRenamed('city','city_csv')
Csv_df2 = Csv_df2.select('id', 'city_csv','state_id','lat','lng')
Csv_df2 = Csv_df2.withColumn('lat', F.col('lat').cast("decimal(8,6)")) \
                    .withColumn('lng', F.col('lng').cast("decimal(9,6)")) \
                    # .withColumn('population', F.col('population').cast('int')) \
                    # .withColumn('density', F.col('density').cast('int'))
# sw_df_joined = sw_df_exploded3.join(Csv_df2.select('city_csv','state_id','id'), (sw_df_exploded3['city'] == Csv_df2['city_csv']) &
#                                     (sw_df_exploded3['state'] == Csv_df2['state_id']), "left").drop('city_csv','state_id', 'city', 'state')
# sw_df_joined = sw_df_joined.withColumn('timestamp', F.current_timestamp())

# sw_df_joined = sw_df_joined.withColumn('ends', F.col('ends').cast("timestamp"))


### USING THE SAME/UGC CODE TO GE THE LOCATIONS
sw_df_joined = sw_df_exploded3.withColumn('timestamp', F.current_timestamp())
sw_df_joined = sw_df_joined.withColumn('ends', F.col('ends').cast("timestamp"))

display(sw_df_joined)

In [0]:
gold_catalog = 'gold'
gold_schema = 'swiftlogix'
gold_table = 'fact_active_hazards'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

sw_df_joined.write \
    .format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable(f'{gold_catalog}.{gold_schema}.{gold_table}')

Csv_df2.write \
    .format('delta') \
    .mode('Overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable(f'{gold_catalog}.{gold_schema}.dim_warehouse_hub')

hazard_type_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("OverwriteSchema", "true") \
    .saveAsTable(f'{gold_catalog}.{gold_schema}.dim_hazard_type')
    
test_same_final.write \
    .format('delta') \
    .mode('overwrite') \
    .option('OverwriteSchema', 'true') \
    .saveAsTable(f'{gold_catalog}.{gold_schema}.dim_same_code_location')

print(datetime.datetime.now())